# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VipS-2004/flyrank1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:

!git clone https://github.com/VipS-2004/flyrank1.git

fatal: destination path 'flyrank1' already exists and is not an empty directory.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: The Age-Freshness Matrix

The paper reports that older content can still perform well when it is recently updated. In particular, the paper identifies a "Refresh Winners" zone where content older than 365 days but updated within 30 days has a health score of 36.98, compared with 39.02 for newer recently updated content.

**Methodology question:** Where does the health-score label come from, and how is it constructed? I would also want to understand whether the comparison controls for differences between clients, topics, and existing search visibility. If the groups have different underlying characteristics, the observed difference may not be caused by freshness alone.

### Finding 2: The Growth Prediction Model

The paper reports that its growth model reaches about 90% accuracy on unseen pages from the same brands and about 75% on brands it has never seen before.

**Methodology question:** Does the validation design fully support the generalization claim? In particular, I would want to confirm that pages from the same client are handled appropriately between training and test sets, and that information from future periods cannot enter the features used for prediction. The separate same-brand and unseen-brand results are useful, but the split design is important for interpreting how well the model transfers to genuinely new clients.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Method

My Week-5 notebook was primarily a signal/correlation analysis rather than a trained predictive model. To make the validation question testable, I use a simple Random Forest Regressor here to predict engagement rate from the Week-5 signal features.

I first evaluate it with a standard random train/test split, then repeat the evaluation using a client-grouped split. The grouped split is more conservative because content from the same client should not appear in both training and test sets when the goal is to understand performance on unseen clients.

The comparison is intended as decision-support and validation evidence, not as proof of generalization.

In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv(
    "/content/flyrank1/data/raw/content_refresh_anonymized.csv"
)


feature_cols = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_traffic_pct",
    "word_count",
    "ctr",
    "avg_position",
    "scroll_rate",
    "trend_pct"
]

model_df = df[feature_cols + ["engagement_rate", "client_id"]].dropna()

X = model_df[feature_cols]
y = model_df["engagement_rate"]



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

random_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

random_mae = mean_absolute_error(y_test, random_pred)
random_r2 = r2_score(y_test, random_pred)





clients = model_df["client_id"].unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_mask = model_df["client_id"].isin(train_clients)
test_mask = model_df["client_id"].isin(test_clients)

X_train_grouped = X[train_mask]
X_test_grouped = X[test_mask]

y_train_grouped = y[train_mask]
y_test_grouped = y[test_mask]

grouped_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

grouped_model.fit(X_train_grouped, y_train_grouped)

grouped_pred = grouped_model.predict(X_test_grouped)

grouped_mae = mean_absolute_error(
    y_test_grouped,
    grouped_pred
)

grouped_r2 = r2_score(
    y_test_grouped,
    grouped_pred
)




comparison = pd.DataFrame({
    "Split": [
        "Random split (before)",
        "Client-grouped split (after)"
    ],
    "MAE": [
        random_mae,
        grouped_mae
    ],
    "R2": [
        random_r2,
        grouped_r2
    ]
})

display(comparison)

,Split,MAE,R2
0,Random split (before),3.064889,0.117171
1,Client-grouped split (after),3.333219,-0.050984


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I checked the final feature set for variables that could directly reveal the target or contain information that would only be available after the prediction point.

The target `engagement_rate` is excluded from the model features. I also excluded derived Week-4 fields such as `baseline_score`, `reason_code`, and `action` because they are based on the heuristic and could create circular information.

The remaining features describe content age, historical traffic, clicks, sessions, CTR, position, scrolling, AI traffic, word count, and trend. These are treated as input signals rather than outcome labels.

A remaining limitation is that some historical traffic features may reflect activity from periods close to the measurement of engagement. A stricter time-based validation would be needed to fully establish that no future information enters the feature set.

In [18]:

target = "engagement_rate"

derived_week4_fields = [
    "baseline_score",
    "reason_code",
    "action"
]

print("Target in feature set:", target in feature_cols)

print("\nWeek-4 derived fields in feature set:")
for col in derived_week4_fields:
    print(f"{col}: {col in feature_cols}")

print("\nFinal model features:")
print(feature_cols)


Target in feature set: False

Week-4 derived fields in feature set:
baseline_score: False
reason_code: False
action: False

Final model features:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_traffic_pct', 'word_count', 'ctr', 'avg_position', 'scroll_rate', 'trend_pct']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim:**  
The model can predict engagement rate accurately and generalize to new content.

**Safer claim:**  
The analysis measured a directional relationship between the selected content signals and engagement rate. Under a random split, the model achieved an R² of 0.117 and MAE of 3.065, while the client-grouped split produced an R² of -0.051 and MAE of 3.333. The grouped result suggests that performance on unseen clients is weaker than the random-split result. Therefore, the current model should be treated as decision-support evidence rather than as proof of reliable generalization.

In [19]:
# Claim rewrite check

print("Original claim:")
print("The model can predict engagement rate accurately and generalize to new content.")

print("\nSafer claim:")
print(
    "The analysis measured directional relationships between selected "
    "content signals and engagement rate. The random split performed "
    "better than the client-grouped split, so the current results should "
    "be treated as decision-support evidence rather than proof of reliable "
    "generalization."
)


Original claim:
The model can predict engagement rate accurately and generalize to new content.

Safer claim:
The analysis measured directional relationships between selected content signals and engagement rate. The random split performed better than the client-grouped split, so the current results should be treated as decision-support evidence rather than proof of reliable generalization.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.